# EuroCrimePulse — Analysis Notebook

**Environment:** Docker container `eurocrimepulse:v2`  
**Data source:** HDFS Gold Star Schema Warehouse (`hdfs://localhost:9000/eurocrimepulse/warehouse`)  
**Schema:** Built by `gold_star_schema.py` — 12 dimension tables + `fact_crime_case`  

> Run `gold_star_schema.py` before executing this notebook to ensure warehouse data exists.


In [ ]:
# EuroCrimePulse — Docker / Standalone Spark setup
# This notebook runs inside the eurocrimepulse Docker container
# or against a Spark cluster with HDFS access.
# Google Colab mount has been replaced with HDFS warehouse path.
import os
print('Environment ready.')


Mounted at /content/drive


In [ ]:
import os

# Warehouse path: HDFS Parquet produced by gold_star_schema.py
# Override with environment variable if needed.
HDFS_BASE = os.getenv(
    'EUROCRIMEPULSE_HDFS_BASE', 'hdfs://localhost:9000/eurocrimepulse'
)
BASE_DIR = os.getenv(
    'STAR_SCHEMA_DIR',
    f'{HDFS_BASE}/warehouse',
)
print(f'Warehouse path: {BASE_DIR}')


['dim_crime_type', 'dim_city', 'dim_geolocation', 'dim_court', 'dim_judge', 'dim_officer', 'dim_sentence_type', 'dim_verdict_type', 'dim_release_reason', 'dim_victim', 'dim_defendant', 'dim_date', 'fact_crime_case', 'crime_star_schema.xlsx']


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EuroCrimePulse") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 4.0.3


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName('EuroCrimePulse-Analysis')
    .config('spark.sql.session.timeZone', 'UTC')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)

TABLES = [
    'dim_crime_type', 'dim_city', 'dim_geolocation', 'dim_court',
    'dim_judge', 'dim_officer', 'dim_sentence_type', 'dim_verdict_type',
    'dim_release_reason', 'dim_victim', 'dim_defendant', 'dim_date',
    'fact_crime_case',
]

print(f'Loading {len(TABLES)} tables from: {BASE_DIR}')
loaded_tables = []
for name in TABLES:
    path = f'{BASE_DIR}/{name}'
    try:
        df = spark.read.parquet(path)
        df.createOrReplaceTempView(name)
        cnt = df.count()
        print(f'  ok {name}: {cnt:,} rows')
        loaded_tables.append(name)
    except Exception as e:
        print(f'  SKIP {name}: {e}')

print(f'Loaded {len(loaded_tables)}/{len(TABLES)} tables successfully')


dim_crime_type             13 rows
dim_city                   16 rows
dim_geolocation       100,000 rows
dim_court              17,015 rows
dim_judge              19,886 rows
dim_officer            29,652 rows
dim_sentence_type           3 rows
dim_verdict_type            3 rows
dim_release_reason          6 rows
dim_victim            100,000 rows
dim_defendant         100,000 rows
dim_date                5,805 rows
fact_crime_case       100,000 rows


## 1. Crime Overview & Trends

Total cases

In [ ]:
spark.sql("""
SELECT
    COUNT(DISTINCT crime_case_key) AS total_cases
FROM fact_crime_case
""").show()

+-----------+
|total_cases|
+-----------+
|     100000|
+-----------+



Number of Crime Types

In [ ]:
spark.sql("""
SELECT
    COUNT(DISTINCT crime_type_key) AS total_crime_types
FROM fact_crime_case
""").show()

+-----------------+
|total_crime_types|
+-----------------+
|               13|
+-----------------+



Top City Cases

In [ ]:
spark.sql("""
SELECT
 c.country_name,
    c.city_name,
    COUNT(DISTINCT f.crime_case_key) AS crime_count
FROM fact_crime_case f
JOIN dim_city c
    ON f.city_key = c.city_key
GROUP BY
    c.city_name,
    c.country_name
ORDER BY crime_count DESC
LIMIT 10
""").show()

+------------+---------+-----------+
|country_name|city_name|crime_count|
+------------+---------+-----------+
|     Ireland|   Dublin|      12536|
|     Belgium| Brussels|      12303|
|       Spain|Barcelona|       6359|
| Netherlands|Amsterdam|       6308|
|       Spain|   Madrid|       6304|
|      Poland|   Krakow|       6291|
|      Poland|   Warsaw|       6243|
| Netherlands|Rotterdam|       6196|
|       Italy|    Milan|       6189|
|       Italy|     Rome|       6186|
+------------+---------+-----------+



Peak Crime Period

In [ ]:
spark.sql("""
SELECT
    d.year,
    d.month,
    d.month_name,
    COUNT(DISTINCT f.crime_case_key) AS crime_count
FROM fact_crime_case f
JOIN dim_date d
    ON f.crime_date_key = d.date_key
GROUP BY
    d.year,
    d.month,
    d.month_name
ORDER BY crime_count DESC
LIMIT 1
""").show()

+----+-----+----------+-----------+
|year|month|month_name|crime_count|
+----+-----+----------+-----------+
|2025|   12|  December|       1816|
+----+-----+----------+-----------+



In [ ]:
spark.sql("""
SELECT
    ROUND(AVG(crime_count), 2) AS avg_monthly_cases
FROM (
    SELECT
        d.year,
        d.month,
        COUNT(DISTINCT f.crime_case_key) AS crime_count
    FROM fact_crime_case f
    JOIN dim_date d
        ON f.crime_date_key = d.date_key
    GROUP BY
        d.year,
        d.month
)
""").show()

+-----------------+
|avg_monthly_cases|
+-----------------+
|          1666.67|
+-----------------+



In [ ]:
spark.sql("""
SELECT
    c.crime_type,
    COUNT(DISTINCT f.crime_case_key) AS crime_count
FROM fact_crime_case f
JOIN dim_crime_type c
    ON f.crime_type_key = c.crime_type_key
GROUP BY
    c.crime_type
ORDER BY crime_count DESC
""").show()

+----------------+-----------+
|      crime_type|crime_count|
+----------------+-----------+
|             Dui|       7889|
|         Bribery|       7764|
|           Theft|       7734|
|       Vandalism|       7734|
|      Cybercrime|       7714|
|       Smuggling|       7699|
|Money Laundering|       7686|
|        Homicide|       7681|
|           Fraud|       7670|
|         Assault|       7668|
| Drug Possession|       7631|
|         Robbery|       7587|
|        Burglary|       7543|
+----------------+-----------+



In [ ]:
spark.sql("""
WITH crime_counts AS (
    SELECT
        c.crime_type,
        COUNT(DISTINCT f.crime_case_key) AS crime_count
    FROM fact_crime_case f
    JOIN dim_crime_type c
        ON f.crime_type_key = c.crime_type_key
    GROUP BY c.crime_type
),
total AS (
    SELECT SUM(crime_count) AS total_cases
    FROM crime_counts
)
SELECT
    crime_type,
    crime_count,
    CONCAT(
        CAST(ROUND(crime_count * 100.0 / total_cases, 2) AS STRING),
        '%'
    ) AS crime_share
FROM crime_counts
CROSS JOIN total
ORDER BY crime_count DESC
""").show()

+----------------+-----------+-----------+
|      crime_type|crime_count|crime_share|
+----------------+-----------+-----------+
|             Dui|       7889|      7.89%|
|         Bribery|       7764|      7.76%|
|           Theft|       7734|      7.73%|
|       Vandalism|       7734|      7.73%|
|      Cybercrime|       7714|      7.71%|
|       Smuggling|       7699|      7.70%|
|Money Laundering|       7686|      7.69%|
|        Homicide|       7681|      7.68%|
|           Fraud|       7670|      7.67%|
|         Assault|       7668|      7.67%|
| Drug Possession|       7631|      7.63%|
|         Robbery|       7587|      7.59%|
|        Burglary|       7543|      7.54%|
+----------------+-----------+-----------+



### Crime Count by Year

In [ ]:
spark.sql("""
SELECT
    d.year,
    COUNT(DISTINCT f.crime_case_key) AS crime_count
FROM fact_crime_case f
JOIN dim_date d ON f.crime_date_key = d.date_key
GROUP BY d.year
ORDER BY d.year
""").show()

+----+-----------+
|year|crime_count|
+----+-----------+
|2021|       6533|
|2022|      20013|
|2023|      20349|
|2024|      20225|
|2025|      20491|
|2026|      12389|
+----+-----------+



### Crime Count by Month within Year

In [ ]:
spark.sql("""
SELECT
    d.year,
    d.month,
    d.month_name,
    COUNT(DISTINCT f.crime_case_key) AS crime_count
FROM fact_crime_case f
JOIN dim_date d ON f.crime_date_key = d.date_key
GROUP BY d.year, d.month, d.month_name
ORDER BY d.year, d.month
""").show(100)

+----+-----+----------+-----------+
|year|month|month_name|crime_count|
+----+-----+----------+-----------+
|2021|    9| September|       1369|
|2021|   10|   October|       1720|
|2021|   11|  November|       1693|
|2021|   12|  December|       1751|
|2022|    1|   January|       1695|
|2022|    2|  February|       1556|
|2022|    3|     March|       1671|
|2022|    4|     April|       1626|
|2022|    5|       May|       1649|
|2022|    6|      June|       1614|
|2022|    7|      July|       1726|
|2022|    8|    August|       1728|
|2022|    9| September|       1644|
|2022|   10|   October|       1710|
|2022|   11|  November|       1687|
|2022|   12|  December|       1707|
|2023|    1|   January|       1693|
|2023|    2|  February|       1583|
|2023|    3|     March|       1686|
|2023|    4|     April|       1676|
|2023|    5|       May|       1776|
|2023|    6|      June|       1719|
|2023|    7|      July|       1754|
|2023|    8|    August|       1684|
|2023|    9| September|     

### Crime Count by Day

In [ ]:
spark.sql("""
SELECT
    d.year,
    d.month,
    d.month_name,
    DAY(d.full_date) AS day,
    d.day_name,
    COUNT(DISTINCT f.crime_case_key) AS crime_count
FROM fact_crime_case f
JOIN dim_date d
    ON f.crime_date_key = d.date_key
GROUP BY
    d.year,
    d.month,
    d.month_name,
    DAY(d.full_date),
    d.day_name
ORDER BY
    d.year,
    d.month,
    day
""").show(100, False)

+----+-----+----------+---+---------+-----------+
|year|month|month_name|day|day_name |crime_count|
+----+-----+----------+---+---------+-----------+
|2021|9    |September |5  |Sunday   |6          |
|2021|9    |September |6  |Monday   |58         |
|2021|9    |September |7  |Tuesday  |55         |
|2021|9    |September |8  |Wednesday|62         |
|2021|9    |September |9  |Thursday |67         |
|2021|9    |September |10 |Friday   |52         |
|2021|9    |September |11 |Saturday |57         |
|2021|9    |September |12 |Sunday   |50         |
|2021|9    |September |13 |Monday   |54         |
|2021|9    |September |14 |Tuesday  |54         |
|2021|9    |September |15 |Wednesday|52         |
|2021|9    |September |16 |Thursday |54         |
|2021|9    |September |17 |Friday   |46         |
|2021|9    |September |18 |Saturday |49         |
|2021|9    |September |19 |Sunday   |50         |
|2021|9    |September |20 |Monday   |52         |
|2021|9    |September |21 |Tuesday  |52         |


In [ ]:
spark.sql("""
SELECT
    d.year,
    d.month,
    d.month_name,
    COUNT(DISTINCT f.crime_case_key) AS crime_count
FROM fact_crime_case f
JOIN dim_date d
    ON f.crime_date_key = d.date_key
GROUP BY
    d.year,
    d.month,
    d.month_name
ORDER BY
    d.year,
    d.month
""").show(100, False)

+----+-----+----------+-----------+
|year|month|month_name|crime_count|
+----+-----+----------+-----------+
|2021|9    |September |1369       |
|2021|10   |October   |1720       |
|2021|11   |November  |1693       |
|2021|12   |December  |1751       |
|2022|1    |January   |1695       |
|2022|2    |February  |1556       |
|2022|3    |March     |1671       |
|2022|4    |April     |1626       |
|2022|5    |May       |1649       |
|2022|6    |June      |1614       |
|2022|7    |July      |1726       |
|2022|8    |August    |1728       |
|2022|9    |September |1644       |
|2022|10   |October   |1710       |
|2022|11   |November  |1687       |
|2022|12   |December  |1707       |
|2023|1    |January   |1693       |
|2023|2    |February  |1583       |
|2023|3    |March     |1686       |
|2023|4    |April     |1676       |
|2023|5    |May       |1776       |
|2023|6    |June      |1719       |
|2023|7    |July      |1754       |
|2023|8    |August    |1684       |
|2023|9    |September |1647 

In [ ]:
spark.sql("""
SELECT
    d.year,
    d.month,
    d.month_name,
    COUNT(DISTINCT f.crime_case_key) AS crime_count
FROM fact_crime_case f
JOIN dim_date d
    ON f.crime_date_key = d.date_key
GROUP BY
    d.year,
    d.month,
    d.month_name
ORDER BY
    d.year,
    d.month
""").show(100, False)

+----+-----+----------+-----------+
|year|month|month_name|crime_count|
+----+-----+----------+-----------+
|2021|9    |September |1369       |
|2021|10   |October   |1720       |
|2021|11   |November  |1693       |
|2021|12   |December  |1751       |
|2022|1    |January   |1695       |
|2022|2    |February  |1556       |
|2022|3    |March     |1671       |
|2022|4    |April     |1626       |
|2022|5    |May       |1649       |
|2022|6    |June      |1614       |
|2022|7    |July      |1726       |
|2022|8    |August    |1728       |
|2022|9    |September |1644       |
|2022|10   |October   |1710       |
|2022|11   |November  |1687       |
|2022|12   |December  |1707       |
|2023|1    |January   |1693       |
|2023|2    |February  |1583       |
|2023|3    |March     |1686       |
|2023|4    |April     |1676       |
|2023|5    |May       |1776       |
|2023|6    |June      |1719       |
|2023|7    |July      |1754       |
|2023|8    |August    |1684       |
|2023|9    |September |1647 

In [ ]:
spark.sql("""
SELECT
    d.year,
    COUNT(DISTINCT f.crime_case_key) AS crime_count
FROM fact_crime_case f
JOIN dim_date d
    ON f.crime_date_key = d.date_key
GROUP BY d.year
ORDER BY d.year
""").show()

+----+-----------+
|year|crime_count|
+----+-----------+
|2021|       6533|
|2022|      20013|
|2023|      20349|
|2024|      20225|
|2025|      20491|
|2026|      12389|
+----+-----------+



In [ ]:
spark.sql("""
SELECT
    d.day_name,
    COUNT(DISTINCT f.crime_case_key) AS crime_count
FROM fact_crime_case f
JOIN dim_date d
    ON f.crime_date_key = d.date_key
GROUP BY d.day_name
ORDER BY crime_count DESC
""").show()

+---------+-----------+
| day_name|crime_count|
+---------+-----------+
| Saturday|      14472|
|   Sunday|      14370|
| Thursday|      14275|
|   Monday|      14269|
|Wednesday|      14261|
|   Friday|      14256|
|  Tuesday|      14097|
+---------+-----------+



In [ ]:
spark.sql("DESCRIBE dim_date").show(100, False)

+----------+---------+-------+
|col_name  |data_type|comment|
+----------+---------+-------+
|full_date |date     |NULL   |
|date_key  |bigint   |NULL   |
|month     |int      |NULL   |
|month_name|string   |NULL   |
|quarter   |int      |NULL   |
|year      |int      |NULL   |
|day_name  |string   |NULL   |
+----------+---------+-------+



### Crime Count by Day Name

In [ ]:
spark.sql("""
SELECT
    d.day_name,
    COUNT(DISTINCT f.crime_case_key) AS crime_count
FROM fact_crime_case f
JOIN dim_date d ON f.crime_date_key = d.date_key
GROUP BY d.day_name
ORDER BY crime_count DESC
""").show()

+---------+-----------+
| day_name|crime_count|
+---------+-----------+
| Saturday|      14472|
|   Sunday|      14370|
| Thursday|      14275|
|   Monday|      14269|
|Wednesday|      14261|
|   Friday|      14256|
|  Tuesday|      14097|
+---------+-----------+



**Dashboard rule:** keep `year`, `month`, `day`, and `day_name` as separate hierarchy fields. Do not concatenate `month_name + year` for the main trend.

## 2. Geographic Distribution

In [ ]:
spark.sql("""
SELECT
    c.city_name,
    c.country_name,
    COUNT(DISTINCT f.crime_case_key) AS crime_count
FROM fact_crime_case f
JOIN dim_city c
    ON f.city_key = c.city_key
GROUP BY
    c.city_name,
    c.country_name
ORDER BY
    crime_count DESC
LIMIT 10
""").show()

+---------+------------+-----------+
|city_name|country_name|crime_count|
+---------+------------+-----------+
|   Dublin|     Ireland|      12536|
| Brussels|     Belgium|      12303|
|Barcelona|       Spain|       6359|
|Amsterdam| Netherlands|       6308|
|   Madrid|       Spain|       6304|
|   Krakow|      Poland|       6291|
|   Warsaw|      Poland|       6243|
|Rotterdam| Netherlands|       6196|
|    Milan|       Italy|       6189|
|     Rome|       Italy|       6186|
+---------+------------+-----------+



In [ ]:
spark.sql("""
SELECT
    c.city_name,
    c.country_name,
    ct.crime_type,
    COUNT(DISTINCT f.crime_case_key) AS crime_count
FROM fact_crime_case f
JOIN dim_city c
    ON f.city_key = c.city_key
JOIN dim_crime_type ct
    ON f.crime_type_key = ct.crime_type_key
GROUP BY
    c.city_name,
    c.country_name,
    ct.crime_type
ORDER BY
    c.city_name,
    crime_count DESC
""").show(100)

+---------+------------+----------------+-----------+
|city_name|country_name|      crime_type|crime_count|
+---------+------------+----------------+-----------+
|Amsterdam| Netherlands|             Dui|        502|
|Amsterdam| Netherlands|        Homicide|        501|
|Amsterdam| Netherlands|       Smuggling|        497|
|Amsterdam| Netherlands|         Bribery|        496|
|Amsterdam| Netherlands|Money Laundering|        492|
|Amsterdam| Netherlands|           Fraud|        492|
|Amsterdam| Netherlands|       Vandalism|        492|
|Amsterdam| Netherlands| Drug Possession|        491|
|Amsterdam| Netherlands|        Burglary|        473|
|Amsterdam| Netherlands|           Theft|        471|
|Amsterdam| Netherlands|         Robbery|        469|
|Amsterdam| Netherlands|         Assault|        468|
|Amsterdam| Netherlands|      Cybercrime|        464|
|Barcelona|       Spain|         Assault|        524|
|Barcelona|       Spain|             Dui|        517|
|Barcelona|       Spain|    

In [ ]:
spark.sql("""
SELECT
    c.city_name,
    c.country_name,
    d.year,
    COUNT(DISTINCT f.crime_case_key) AS crime_count
FROM fact_crime_case f
JOIN dim_city c
    ON f.city_key = c.city_key
JOIN dim_date d
    ON f.crime_date_key = d.date_key
GROUP BY
    c.city_name,
    c.country_name,
    d.year
ORDER BY
    c.city_name,
    d.year
""").show(100)

+---------+------------+----+-----------+
|city_name|country_name|year|crime_count|
+---------+------------+----+-----------+
|Amsterdam| Netherlands|2021|        431|
|Amsterdam| Netherlands|2022|       1254|
|Amsterdam| Netherlands|2023|       1275|
|Amsterdam| Netherlands|2024|       1308|
|Amsterdam| Netherlands|2025|       1274|
|Amsterdam| Netherlands|2026|        766|
|Barcelona|       Spain|2021|        409|
|Barcelona|       Spain|2022|       1254|
|Barcelona|       Spain|2023|       1318|
|Barcelona|       Spain|2024|       1246|
|Barcelona|       Spain|2025|       1268|
|Barcelona|       Spain|2026|        864|
|   Berlin|     Germany|2021|        258|
|   Berlin|     Germany|2022|        857|
|   Berlin|     Germany|2023|        772|
|   Berlin|     Germany|2024|        793|
|   Berlin|     Germany|2025|        883|
|   Berlin|     Germany|2026|        521|
| Brussels|     Belgium|2021|        854|
| Brussels|     Belgium|2022|       2525|
| Brussels|     Belgium|2023|     

In [ ]:
spark.sql("""
WITH city_yearly AS (
    SELECT
        c.city_name,
        c.country_name,
        d.year,
        COUNT(DISTINCT f.crime_case_key) AS crime_count
    FROM fact_crime_case f
    JOIN dim_city c
        ON f.city_key = c.city_key
    JOIN dim_date d
        ON f.crime_date_key = d.date_key
    GROUP BY
        c.city_name,
        c.country_name,
        d.year
),

city_change AS (
    SELECT
        city_name,
        country_name,
        year,
        crime_count,
        LAG(crime_count) OVER (
            PARTITION BY city_name, country_name
            ORDER BY year
        ) AS previous_year_cases
    FROM city_yearly
)

SELECT
    city_name,
    country_name,
    year,
    crime_count,
    previous_year_cases,
    ROUND(
        (crime_count - previous_year_cases) * 100.0
        / previous_year_cases,
        2
    ) AS change_pct
FROM city_change
WHERE previous_year_cases IS NOT NULL
ORDER BY change_pct DESC
LIMIT 10
""").show()

+---------+------------+----+-----------+-------------------+----------+
|city_name|country_name|year|crime_count|previous_year_cases|change_pct|
+---------+------------+----+-----------+-------------------+----------+
|     Lyon|      France|2022|        851|                254|    235.04|
|   Berlin|     Germany|2022|        857|                258|    232.17|
|   Dublin|     Ireland|2022|       2526|                790|    219.75|
|  Hamburg|     Germany|2022|        859|                270|    218.15|
|Marseille|      France|2022|        842|                267|    215.36|
|   Warsaw|      Poland|2022|       1209|                387|    212.40|
|   Krakow|      Poland|2022|       1262|                404|    212.38|
|Rotterdam| Netherlands|2022|       1262|                404|    212.38|
|   Munich|     Germany|2022|        847|                275|    208.00|
|    Paris|      France|2022|        817|                266|    207.14|
+---------+------------+----+-----------+----------

In [ ]:
spark.sql("""
WITH city_yearly AS (
    SELECT
        c.city_name,
        c.country_name,
        d.year,
        COUNT(DISTINCT f.crime_case_key) AS crime_count
    FROM fact_crime_case f
    JOIN dim_city c
        ON f.city_key = c.city_key
    JOIN dim_date d
        ON f.crime_date_key = d.date_key
    GROUP BY
        c.city_name,
        c.country_name,
        d.year
),

city_change AS (
    SELECT
        city_name,
        country_name,
        year,
        crime_count,
        LAG(crime_count) OVER (
            PARTITION BY city_name, country_name
            ORDER BY year
        ) AS previous_year_cases
    FROM city_yearly
)

SELECT
    city_name,
    country_name,
    year,
    crime_count,
    previous_year_cases,
    ROUND(
        (crime_count - previous_year_cases) * 100.0
        / previous_year_cases,
        2
    ) AS change_pct
FROM city_change
WHERE previous_year_cases IS NOT NULL
ORDER BY ABS(change_pct) DESC
LIMIT 10
""").show()

+---------+------------+----+-----------+-------------------+----------+
|city_name|country_name|year|crime_count|previous_year_cases|change_pct|
+---------+------------+----+-----------+-------------------+----------+
|     Lyon|      France|2022|        851|                254|    235.04|
|   Berlin|     Germany|2022|        857|                258|    232.17|
|   Dublin|     Ireland|2022|       2526|                790|    219.75|
|  Hamburg|     Germany|2022|        859|                270|    218.15|
|Marseille|      France|2022|        842|                267|    215.36|
|   Warsaw|      Poland|2022|       1209|                387|    212.40|
|   Krakow|      Poland|2022|       1262|                404|    212.38|
|Rotterdam| Netherlands|2022|       1262|                404|    212.38|
|   Munich|     Germany|2022|        847|                275|    208.00|
|    Paris|      France|2022|        817|                266|    207.14|
+---------+------------+----+-----------+----------

## 3. Judiciary & Verdicts

In [ ]:
spark.sql("""
SELECT
    v.verdict_type,
    COUNT(DISTINCT f.crime_case_key) AS case_count
FROM fact_crime_case f
JOIN dim_verdict_type v
    ON f.verdict_type_key = v.verdict_type_key
GROUP BY
    v.verdict_type
ORDER BY
    case_count DESC
""").show()

+------------+----------+
|verdict_type|case_count|
+------------+----------+
|      Guilty|     59926|
|  Not Guilty|     20078|
|     Unknown|     19996|
+------------+----------+



In [ ]:
spark.sql("""
WITH verdict_counts AS (
    SELECT
        v.verdict_type,
        COUNT(DISTINCT f.crime_case_key) AS case_count
    FROM fact_crime_case f
    JOIN dim_verdict_type v
        ON f.verdict_type_key = v.verdict_type_key
    GROUP BY
        v.verdict_type
),
total AS (
    SELECT SUM(case_count) AS total_cases
    FROM verdict_counts
)
SELECT
    verdict_type,
    case_count,
    ROUND(case_count * 100.0 / total_cases, 2) AS verdict_share_pct
FROM verdict_counts
CROSS JOIN total
ORDER BY
    case_count DESC
""").show()

+------------+----------+-----------------+
|verdict_type|case_count|verdict_share_pct|
+------------+----------+-----------------+
|      Guilty|     59926|            59.93|
|  Not Guilty|     20078|            20.08|
|     Unknown|     19996|            20.00|
+------------+----------+-----------------+



In [ ]:
spark.sql("""
SELECT
    ct.crime_type,
    vt.verdict_type,
    COUNT(DISTINCT f.crime_case_key) AS case_count
FROM fact_crime_case f
JOIN dim_crime_type ct
    ON f.crime_type_key = ct.crime_type_key
JOIN dim_verdict_type vt
    ON f.verdict_type_key = vt.verdict_type_key
GROUP BY
    ct.crime_type,
    vt.verdict_type
ORDER BY
    ct.crime_type,
    case_count DESC
""").show(100)

+----------------+------------+----------+
|      crime_type|verdict_type|case_count|
+----------------+------------+----------+
|         Assault|      Guilty|      4535|
|         Assault|     Unknown|      1589|
|         Assault|  Not Guilty|      1544|
|         Bribery|      Guilty|      4669|
|         Bribery|  Not Guilty|      1554|
|         Bribery|     Unknown|      1541|
|        Burglary|      Guilty|      4588|
|        Burglary|  Not Guilty|      1481|
|        Burglary|     Unknown|      1474|
|      Cybercrime|      Guilty|      4649|
|      Cybercrime|     Unknown|      1563|
|      Cybercrime|  Not Guilty|      1502|
| Drug Possession|      Guilty|      4570|
| Drug Possession|     Unknown|      1531|
| Drug Possession|  Not Guilty|      1530|
|             Dui|      Guilty|      4715|
|             Dui|  Not Guilty|      1620|
|             Dui|     Unknown|      1554|
|           Fraud|      Guilty|      4613|
|           Fraud|  Not Guilty|      1536|
|          

In [ ]:
spark.sql("""
SELECT
    c.court_name,
    COUNT(DISTINCT f.crime_case_key) AS case_count
FROM fact_crime_case f
JOIN dim_court c
    ON f.court_key = c.court_key
GROUP BY
    c.court_name
ORDER BY
    case_count DESC
LIMIT 10
""").show()

+--------------------+----------+
|          court_name|case_count|
+--------------------+----------+
|Lake William Supr...|        73|
|East Karenchester...|        57|
|New Elizabeth Reg...|        55|
|New Michael Supre...|        54|
|East Neil Distric...|        51|
|North Michael Sup...|        48|
|South David Distr...|        48|
|Port Susan Suprem...|        47|
|New Michael Regio...|        46|
|Jamesmouth Region...|        46|
+--------------------+----------+



In [ ]:
spark.sql("""
SELECT
    c.court_name,
    d.year,
    COUNT(DISTINCT f.crime_case_key) AS case_count
FROM fact_crime_case f
JOIN dim_court c
    ON f.court_key = c.court_key
JOIN dim_date d
    ON f.verdict_date_key = d.date_key
GROUP BY
    c.court_name,
    d.year
ORDER BY
    c.court_name,
    d.year
""").show(100)

+--------------------+----+----------+
|          court_name|year|case_count|
+--------------------+----+----------+
|Aaronberg Regiona...|2022|         1|
|Aaronberg Regiona...|2023|         2|
|Aaronberg Regiona...|2024|         1|
|Aaronberg Regiona...|2025|         2|
|Aaronberg Regiona...|2026|         1|
|Aaronberg Regiona...|2027|         1|
|Aaronborough Regi...|2021|         1|
|Aaronborough Regi...|2022|         4|
|Aaronborough Regi...|2023|         3|
|Aaronborough Regi...|2024|         5|
|Aaronborough Regi...|2025|         6|
|Aaronborough Regi...|2026|         4|
|Aaronborough Regi...|2027|         1|
|Aaronfort Appella...|2024|         1|
|Aaronfort Appella...|2025|         1|
|Aaronfort Appella...|2026|         4|
|Aaronfurt Appella...|2022|         2|
|Aaronfurt Appella...|2023|         1|
|Aaronfurt Appella...|2024|         3|
|Aaronfurt Appella...|2025|         3|
|Aaronfurt Appella...|2026|         1|
|Aaronhaven Appell...|2021|         1|
|Aaronhaven Appell...|202

In [ ]:
spark.sql("""
SELECT
    st.sentence_type,
    COUNT(DISTINCT f.crime_case_key) AS case_count
FROM fact_crime_case f
JOIN dim_sentence_type st
    ON f.sentence_type_key = st.sentence_type_key
GROUP BY
    st.sentence_type
ORDER BY
    case_count DESC
""").show()

+-------------+----------+
|sentence_type|case_count|
+-------------+----------+
| Imprisonment|     20071|
|    Probation|     20041|
|         Fine|     19814|
+-------------+----------+



In [ ]:
spark.sql("""
SELECT
    ct.crime_type,
    ROUND(AVG(f.sentence_duration), 2) AS avg_sentence_duration,
    COUNT(DISTINCT f.crime_case_key) AS sentenced_cases
FROM fact_crime_case f
JOIN dim_crime_type ct
    ON f.crime_type_key = ct.crime_type_key
WHERE
    f.sentence_duration IS NOT NULL
    AND f.is_duration_invalid_flag = false
GROUP BY
    ct.crime_type
ORDER BY
    avg_sentence_duration DESC
""").show()

+----------------+---------------------+---------------+
|      crime_type|avg_sentence_duration|sentenced_cases|
+----------------+---------------------+---------------+
|Money Laundering|              1316.98|           1353|
| Drug Possession|              1310.85|           1292|
|      Cybercrime|               1310.2|           1383|
|           Fraud|              1296.83|           1290|
|       Smuggling|              1288.62|           1394|
|         Robbery|              1278.86|           1326|
|             Dui|              1274.43|           1367|
|       Vandalism|              1252.43|           1384|
|           Theft|              1250.14|           1416|
|        Homicide|               1235.6|           1313|
|        Burglary|               1227.8|           1325|
|         Bribery|              1222.78|           1365|
|         Assault|              1217.52|           1271|
+----------------+---------------------+---------------+



In [ ]:
spark.sql("""
SELECT
    ct.crime_type,

    COUNT(DISTINCT f.crime_case_key) AS case_count,

    ROUND(
        AVG(
            DATEDIFF(
                verdict_date.full_date,
                crime_date.full_date
            )
        ),
        2
    ) AS avg_days_to_verdict

FROM fact_crime_case f

JOIN dim_crime_type ct
    ON f.crime_type_key = ct.crime_type_key

JOIN dim_date crime_date
    ON f.crime_date_key = crime_date.date_key

JOIN dim_date verdict_date
    ON f.verdict_date_key = verdict_date.date_key

WHERE
    f.verdict_date_key IS NOT NULL
    AND f.crime_date_key IS NOT NULL

GROUP BY
    ct.crime_type

ORDER BY
    avg_days_to_verdict DESC
""").show()

+----------------+----------+-------------------+
|      crime_type|case_count|avg_days_to_verdict|
+----------------+----------+-------------------+
|      Cybercrime|      7714|             209.98|
|           Theft|      7734|             209.16|
|       Smuggling|      7699|             208.51|
|         Robbery|      7587|              208.4|
|        Burglary|      7543|             208.29|
|         Assault|      7668|             208.03|
|        Homicide|      7681|             207.59|
|Money Laundering|      7686|             207.42|
|         Bribery|      7764|              207.2|
| Drug Possession|      7631|             206.53|
|             Dui|      7889|              205.9|
|       Vandalism|      7734|              205.2|
|           Fraud|      7670|             205.03|
+----------------+----------+-------------------+



In [ ]:
spark.sql("""
SELECT
    j.judge_name,
    c.court_name,

    COUNT(DISTINCT f.crime_case_key) AS case_count,

    ROUND(
        AVG(
            DATEDIFF(
                vd.full_date,
                cd.full_date
            )
        ),
        2
    ) AS avg_days_to_verdict

FROM fact_crime_case f

JOIN dim_judge j
    ON f.judge_key = j.judge_key

JOIN dim_court c
    ON f.court_key = c.court_key

JOIN dim_date cd
    ON f.crime_date_key = cd.date_key

JOIN dim_date vd
    ON f.verdict_date_key = vd.date_key

WHERE
    f.crime_date_key IS NOT NULL
    AND f.verdict_date_key IS NOT NULL
    AND vd.full_date >= cd.full_date

GROUP BY
    j.judge_name,
    c.court_name

ORDER BY
    avg_days_to_verdict DESC
""").show(100)

+--------------------+--------------------+----------+-------------------+
|          judge_name|          court_name|case_count|avg_days_to_verdict|
+--------------------+--------------------+----------+-------------------+
|   Richard Gallagher|Port Tracyborough...|         1|              396.0|
|       Todd Williams|Lake Patrick Appe...|         1|              395.0|
|     Joseph Marshall|New Katieview Sup...|         2|              394.5|
|      Alexandra Mann|Lake Gilbert Regi...|         2|              393.0|
|      Eduardo Garcia|New Julie Regiona...|         1|              393.0|
|         Donna Moore|Cynthiashire Regi...|         1|              389.0|
|    Katherine Weaver|Rosetown Supreme ...|         2|              387.0|
|       Arthur Taylor|East Christine Ap...|         2|              386.0|
|      Melinda Sutton|Port Randy Suprem...|         3|              380.0|
|        Paul Perkins|Kerriland Appella...|         1|              380.0|
|         James Welch|Tam

## 4. Corrections & Demographics

In [ ]:
spark.sql("""
SELECT
    ct.crime_type,
    COUNT(DISTINCT f.crime_case_key) AS imprisoned_cases
FROM fact_crime_case f
JOIN dim_crime_type ct
    ON f.crime_type_key = ct.crime_type_key
WHERE
    f.imprisonment_start_date_key IS NOT NULL
GROUP BY
    ct.crime_type
ORDER BY
    imprisoned_cases DESC
""").show()

+----------------+----------------+
|      crime_type|imprisoned_cases|
+----------------+----------------+
|           Theft|            1613|
|       Smuggling|            1602|
|      Cybercrime|            1594|
|       Vandalism|            1583|
|             Dui|            1563|
|         Bribery|            1554|
|Money Laundering|            1544|
|        Homicide|            1524|
|         Robbery|            1519|
|        Burglary|            1514|
|           Fraud|            1510|
| Drug Possession|            1505|
|         Assault|            1446|
+----------------+----------------+



In [ ]:
spark.sql("""
WITH crime_stats AS (
    SELECT
        ct.crime_type,

        COUNT(DISTINCT f.crime_case_key) AS total_cases,

        COUNT(DISTINCT CASE
            WHEN f.imprisonment_start_date_key IS NOT NULL
            THEN f.crime_case_key
        END) AS imprisoned_cases

    FROM fact_crime_case f

    JOIN dim_crime_type ct
        ON f.crime_type_key = ct.crime_type_key

    GROUP BY ct.crime_type
)

SELECT
    crime_type,
    total_cases,
    imprisoned_cases,

    ROUND(
        CAST(imprisoned_cases AS DOUBLE)
        / CAST(total_cases AS DOUBLE)
        * 100,
        2
    ) AS imprisonment_rate_pct

FROM crime_stats

ORDER BY imprisonment_rate_pct DESC
""").show()

+----------------+-----------+----------------+---------------------+
|      crime_type|total_cases|imprisoned_cases|imprisonment_rate_pct|
+----------------+-----------+----------------+---------------------+
|           Theft|       7734|            1613|                20.86|
|       Smuggling|       7699|            1602|                20.81|
|      Cybercrime|       7714|            1594|                20.66|
|       Vandalism|       7734|            1583|                20.47|
|Money Laundering|       7686|            1544|                20.09|
|        Burglary|       7543|            1514|                20.07|
|         Robbery|       7587|            1519|                20.02|
|         Bribery|       7764|            1554|                20.02|
|        Homicide|       7681|            1524|                19.84|
|             Dui|       7889|            1563|                19.81|
| Drug Possession|       7631|            1505|                19.72|
|           Fraud|  

In [ ]:
spark.sql("""
SELECT
    COUNT(DISTINCT crime_case_key) AS imprisoned_cases
FROM fact_crime_case
WHERE imprisonment_start_date_key IS NOT NULL
""").show()

+----------------+
|imprisoned_cases|
+----------------+
|           20071|
+----------------+



In [ ]:
spark.sql("""
SELECT
    ROUND(
        AVG(sentence_duration),
        2
    ) AS avg_sentence_duration
FROM fact_crime_case
WHERE
    sentence_duration IS NOT NULL
    AND is_duration_invalid_flag = false
    AND imprisonment_start_date_key IS NOT NULL
""").show()

+---------------------+
|avg_sentence_duration|
+---------------------+
|              1268.03|
+---------------------+



In [ ]:
spark.sql("""
SELECT
    ct.crime_type,
    COUNT(DISTINCT f.crime_case_key) AS imprisoned_cases,
    ROUND(
        AVG(f.sentence_duration),
        2
    ) AS avg_sentence_duration
FROM fact_crime_case f
JOIN dim_crime_type ct
    ON f.crime_type_key = ct.crime_type_key
WHERE
    f.sentence_duration IS NOT NULL
    AND f.is_duration_invalid_flag = false
    AND f.imprisonment_start_date_key IS NOT NULL
GROUP BY
    ct.crime_type
ORDER BY
    avg_sentence_duration DESC
""").show()

+----------------+----------------+---------------------+
|      crime_type|imprisoned_cases|avg_sentence_duration|
+----------------+----------------+---------------------+
|Money Laundering|            1353|              1316.98|
| Drug Possession|            1292|              1310.85|
|      Cybercrime|            1383|               1310.2|
|           Fraud|            1290|              1296.83|
|       Smuggling|            1394|              1288.62|
|         Robbery|            1326|              1278.86|
|             Dui|            1367|              1274.43|
|       Vandalism|            1384|              1252.43|
|           Theft|            1416|              1250.14|
|        Homicide|            1313|               1235.6|
|        Burglary|            1325|               1227.8|
|         Bribery|            1365|              1222.78|
|         Assault|            1271|              1217.52|
+----------------+----------------+---------------------+



In [ ]:
spark.sql("""
SELECT
    rr.release_reason,
    COUNT(DISTINCT f.crime_case_key) AS released_cases
FROM fact_crime_case f
JOIN dim_release_reason rr
    ON f.release_reason_key = rr.release_reason_key
WHERE
    f.release_date_key IS NOT NULL
GROUP BY
    rr.release_reason
ORDER BY
    released_cases DESC
""").show()

+--------------------+--------------+
|      release_reason|released_cases|
+--------------------+--------------+
|        Early Parole|          3204|
|            Pardoned|          3143|
|    Released on Bail|          3119|
|   Sentence Commuted|          3098|
|    Served Full Term|          3053|
|Escaped (Recaptured)|          3020|
+--------------------+--------------+



In [ ]:
spark.sql("""
SELECT
    ct.crime_type,
    rr.release_reason,
    COUNT(DISTINCT f.crime_case_key) AS released_cases
FROM fact_crime_case f
JOIN dim_crime_type ct
    ON f.crime_type_key = ct.crime_type_key
JOIN dim_release_reason rr
    ON f.release_reason_key = rr.release_reason_key
WHERE
    f.release_date_key IS NOT NULL
GROUP BY
    ct.crime_type,
    rr.release_reason
ORDER BY
    ct.crime_type,
    released_cases DESC
""").show(100)

+----------------+--------------------+--------------+
|      crime_type|      release_reason|released_cases|
+----------------+--------------------+--------------+
|         Assault|            Pardoned|           240|
|         Assault|   Sentence Commuted|           230|
|         Assault|        Early Parole|           229|
|         Assault|    Served Full Term|           228|
|         Assault|    Released on Bail|           225|
|         Assault|Escaped (Recaptured)|           198|
|         Bribery|    Released on Bail|           278|
|         Bribery|            Pardoned|           254|
|         Bribery|        Early Parole|           253|
|         Bribery|   Sentence Commuted|           231|
|         Bribery|Escaped (Recaptured)|           226|
|         Bribery|    Served Full Term|           216|
|        Burglary|    Released on Bail|           245|
|        Burglary|            Pardoned|           235|
|        Burglary|        Early Parole|           234|
|        B

In [ ]:
spark.sql("""
SELECT
    d.year,
    d.month,
    d.month_name,
    COUNT(DISTINCT f.crime_case_key) AS released_cases
FROM fact_crime_case f
JOIN dim_date d
    ON f.release_date_key = d.date_key
GROUP BY
    d.year,
    d.month,
    d.month_name
ORDER BY
    d.year,
    d.month
""").show(100)

+----+-----+----------+--------------+
|year|month|month_name|released_cases|
+----+-----+----------+--------------+
|2021|   10|   October|             1|
|2021|   11|  November|             5|
|2021|   12|  December|            12|
|2022|    1|   January|            15|
|2022|    2|  February|            31|
|2022|    3|     March|            38|
|2022|    4|     April|            43|
|2022|    5|       May|            49|
|2022|    6|      June|            65|
|2022|    7|      July|            92|
|2022|    8|    August|           100|
|2022|    9| September|           116|
|2022|   10|   October|           126|
|2022|   11|  November|           127|
|2022|   12|  December|           157|
|2023|    1|   January|           144|
|2023|    2|  February|           145|
|2023|    3|     March|           168|
|2023|    4|     April|           182|
|2023|    5|       May|           171|
|2023|    6|      June|           179|
|2023|    7|      July|           189|
|2023|    8|    August|  

In [ ]:
spark.sql("""
SELECT
    ROUND(
        AVG(
            DATEDIFF(
                release_date.full_date,
                imprisonment_date.full_date
            )
        ),
        2
    ) AS avg_days_in_prison
FROM fact_crime_case f

JOIN dim_date imprisonment_date
    ON f.imprisonment_start_date_key = imprisonment_date.date_key

JOIN dim_date release_date
    ON f.release_date_key = release_date.date_key

WHERE
    f.imprisonment_start_date_key IS NOT NULL
    AND f.release_date_key IS NOT NULL
    AND release_date.full_date >= imprisonment_date.full_date
""").show()

+------------------+
|avg_days_in_prison|
+------------------+
|           1268.03|
+------------------+



In [ ]:
spark.sql("""
SELECT
    ct.crime_type,

    ROUND(AVG(v.victim_age), 2) AS avg_victim_age,

    ROUND(AVG(d.defendant_age), 2) AS avg_defendant_age,

    COUNT(DISTINCT f.crime_case_key) AS case_count

FROM fact_crime_case f

JOIN dim_crime_type ct
    ON f.crime_type_key = ct.crime_type_key

JOIN dim_victim v
    ON f.victim_key = v.victim_key

JOIN dim_defendant d
    ON f.defendant_key = d.defendant_key

WHERE
    v.victim_age IS NOT NULL
    AND d.defendant_age IS NOT NULL

GROUP BY
    ct.crime_type

ORDER BY
    case_count DESC
""").show(100)

+----------------+--------------+-----------------+----------+
|      crime_type|avg_victim_age|avg_defendant_age|case_count|
+----------------+--------------+-----------------+----------+
|      Cybercrime|          48.0|            47.57|      1970|
|         Assault|         48.83|            48.58|      1963|
|       Vandalism|         47.87|            47.72|      1960|
|             Dui|         47.53|             48.1|      1956|
|       Smuggling|         47.94|            48.85|      1946|
| Drug Possession|         48.81|            47.74|      1941|
|Money Laundering|          48.6|            47.99|      1922|
|           Fraud|         48.39|            47.75|      1921|
|        Homicide|         47.87|            47.84|      1912|
|           Theft|         48.34|            46.92|      1911|
|         Bribery|         47.66|            47.95|      1899|
|         Robbery|         47.75|            47.77|      1897|
|        Burglary|         48.11|            48.36|    

In [ ]:
spark.sql("""
SELECT
    COUNT(DISTINCT crime_case_key) AS active_imprisonment_cases
FROM fact_crime_case
WHERE
    imprisonment_start_date_key IS NOT NULL
    AND release_date_key IS NULL
""").show()

+-------------------------+
|active_imprisonment_cases|
+-------------------------+
|                     1434|
+-------------------------+



## 5. Bail & Fine Analysis

The original notebook did not contain a bail/fine query. First check whether these fields exist in `fact_crime_case`.

In [ ]:
fact_columns = spark.table("fact_crime_case").columns
print("bail_amount present:", "bail_amount" in fact_columns)
print("fine_amount present:", "fine_amount" in fact_columns)
print("Financial columns:", [c for c in fact_columns if "bail" in c.lower() or "fine" in c.lower()])

bail_amount present: False
fine_amount present: False
Financial columns: []


In [ ]:
cols = {c.lower() for c in spark.table("fact_crime_case").columns}

if {"bail_amount", "fine_amount"}.issubset(cols):
    spark.sql("""
SELECT
    ct.crime_type,
    ROUND(AVG(CASE WHEN f.bail_amount >= 0 THEN f.bail_amount END), 2) AS avg_bail,
    ROUND(AVG(CASE WHEN f.fine_amount >= 0 THEN f.fine_amount END), 2) AS avg_fine,
    COUNT(DISTINCT CASE WHEN f.bail_amount >= 0 THEN f.crime_case_key END) AS bail_cases,
    COUNT(DISTINCT CASE WHEN f.fine_amount >= 0 THEN f.crime_case_key END) AS fine_cases
FROM fact_crime_case f
JOIN dim_crime_type ct ON f.crime_type_key = ct.crime_type_key
GROUP BY ct.crime_type
ORDER BY ct.crime_type
""").show(100)
else:
    print("bail_amount/fine_amount are not present in fact_crime_case, so the financial analysis cannot run yet.")

bail_amount/fine_amount are not present in fact_crime_case, so the financial analysis cannot run yet.


In [ ]:
# !pip install -q streamlit  (handled by Docker image — uncomment if running locally)
# In the Docker container these are pre-installed.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 92.5 MB/s eta 0:00:00


In [ ]:
# !pip install -q pyngrok  (handled by Docker image — uncomment if running locally)
# In the Docker container these are pre-installed.


In [99]:
# google.colab file upload removed — not available in Docker.
# Dashboard file is already at /opt/eurocrimepulse/streamlite_Dashboards.py
print('Dashboard file: /opt/eurocrimepulse/streamlite_Dashboards.py')


Saving streamlite_Dashboards.py to streamlite_Dashboards.py


In [100]:
# ngrok tunnel removed for production deployment.
# Streamlit runs on port 8501 inside the container.
# Access the dashboard at: http://localhost:8501
# (or via Docker port mapping: -p 8501:8501)
print('Dashboard: streamlit run /opt/eurocrimepulse/streamlite_Dashboards.py')


In [101]:
import os
# STAR_SCHEMA_DIR is already set from BASE_DIR above.
# Uncomment and modify if running with a local parquet copy:
# os.environ['STAR_SCHEMA_DIR'] = '/data/eurocrimepulse/warehouse'
print('STAR_SCHEMA_DIR:', os.getenv('STAR_SCHEMA_DIR', BASE_DIR))


In [102]:
# Process management not needed inside Docker container.
# Airflow/supervisor handles process lifecycle.
print('Process lifecycle managed by Docker / Airflow.')


In [103]:
# Process management not needed inside Docker container.
# Airflow/supervisor handles process lifecycle.
print('Process lifecycle managed by Docker / Airflow.')


^C


In [104]:
# Start the dashboard (run from terminal, not this notebook):
# streamlit run /opt/eurocrimepulse/streamlite_Dashboards.py \
#     --server.port 8501 --server.address 0.0.0.0
print('Start dashboard: streamlit run /opt/eurocrimepulse/streamlite_Dashboards.py')


In [105]:
# ngrok tunnel removed for production deployment.
# Streamlit runs on port 8501 inside the container.
# Access the dashboard at: http://localhost:8501
# (or via Docker port mapping: -p 8501:8501)
print('Dashboard: streamlit run /opt/eurocrimepulse/streamlite_Dashboards.py')


NgrokTunnel: "https://aspect-unrelated-thirsty.ngrok-free.dev" -> "http://localhost:8501"


## 6. Dashboard Design Rules

- Monthly trend: X-axis = Jan–Dec; Year = separate series/filter.
- Use `month` for sorting and `month_name` only for display.
- Aggregate before plotting because the fact table is large.
- Use Top-N for high-cardinality dimensions such as courts/judges/cities.
- Put complex charts vertically rather than shrinking them side-by-side.
- Streamlit should reuse the SQL logic above so dashboard numbers stay consistent with the notebook.